# 🎵 Next-Gen Hybrid Music Recommender System
### Blending Spotify Acoustic Features, Latent Collaborative SVD, and Popularity Signals

**Author:** Vasu | **Dataset:** Spotify & YouTube Audio Features (20,000+ tracks)

---

## 🎯 Architecture Overview

Conventional recommendation systems often suffer from two major extremes:
1. **Pure Collaborative Filtering:** Suffers from severe **cold-start**, fails on niche/new songs, and requires dense user interaction histories.
2. **Pure Content-Based Filtering:** Tends to create **echo chambers** (recommending only the same artist or sub-genre) and lacks discovery serendipity.

This project implements a **Tri-Brid Recommendation Engine**:
$$\text{Score}(u, t) = \alpha \cdot \text{AcousticSim}(\mathbf{x}_{\text{seed}}, \mathbf{x}_t) + \beta \cdot \text{CollabScore}(u, t) + \gamma \cdot \text{Popularity}(t)$$

```
                                 ┌───────────────────────────────┐
                                 │      Input: Seed Track        │
                                 └──────────────┬────────────────┘
                                                │
                 ┌──────────────────────────────┼──────────────────────────────┐
                 ▼                              ▼                              ▼
   ┌───────────────────────────┐  ┌───────────────────────────┐  ┌───────────────────────────┐
   │    Content-Based Engine   │  │   Collaborative Engine    │  │     Popularity Signal     │
   │  - 9 Spotify Audio Feats  │  │   - Realistic User Personas│  │   - Log(Streams + Views)  │
   │  - Cosine Distance & KNN  │  │   - TruncatedSVD Latent   │  │   - Global Virality Bias  │
   └─────────────┬─────────────┘  └─────────────┬─────────────┘  └─────────────┬─────────────┘
                 │ (Weight: α)                  │ (Weight: β)                  │ (Weight: γ)
                 └──────────────────────────────┼──────────────────────────────┘
                                                ▼
                                 ┌───────────────────────────────┐
                                 │   Hybrid Scoring & Ranking    │
                                 │    (Optional Vibe Filter)     │
                                 └──────────────┬────────────────┘
                                                ▼
                                 ┌───────────────────────────────┐
                                 │ Top-N Personalized Playlist   │
                                 └───────────────────────────────┘
```


## 1. Setup & Data Loading


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix

# Configure visualization style
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("Dependencies successfully imported!")


In [ ]:
# Load the Spotify & YouTube dataset
df = pd.read_csv('Spotify_Youtube.csv')
print(f"Raw dataset shape: {df.shape}")
df.head(3)


## 2. Data Cleaning & Feature Engineering


In [ ]:
# Define core Spotify acoustic features
AUDIO_FEATURES = [
    'Danceability', 'Energy', 'Loudness', 'Speechiness',
    'Acousticness', 'Instrumentalness', 'Liveness', 'Valence', 'Tempo'
]

# Keep relevant metadata and audio features
meta_cols = ['Artist', 'Track', 'Album', 'Url_spotify', 'Url_youtube', 'Views', 'Likes', 'Stream']
df = df.dropna(subset=['Artist', 'Track'] + AUDIO_FEATURES).copy()

for col in ['Views', 'Likes', 'Stream']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Deduplicate artist + track pairs
df = df.drop_duplicates(subset=['Artist', 'Track']).reset_index(drop=True)

# Normalize audio features to [0, 1] range for metric distance calculation
scaler = MinMaxScaler()
norm_cols = [f'norm_{col}' for col in AUDIO_FEATURES]
df[norm_cols] = scaler.fit_transform(df[AUDIO_FEATURES])

# Compute a smooth logarithmic composite popularity score
log_views = np.log1p(df['Views'])
log_likes = np.log1p(df['Likes'])
log_stream = np.log1p(df['Stream'])
raw_pop = 0.4 * log_stream + 0.4 * log_views + 0.2 * log_likes
df['popularity_score'] = (raw_pop - raw_pop.min()) / (raw_pop.max() - raw_pop.min() + 1e-9)

print(f"Cleaned unique tracks: {len(df)}")
df[['Artist', 'Track', 'popularity_score'] + norm_cols[:4]].head()


## 3. Exploratory Data Analysis & Sonic Landscapes


In [ ]:
# Plot 1: Audio Feature Distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
features_to_plot = ['Danceability', 'Energy', 'Valence', 'Acousticness', 'Loudness', 'popularity_score']
colors = ['#38bdf8', '#f59e0b', '#10b981', '#ec4899', '#8b5cf6', '#6366f1']

for ax, feat, color in zip(axes.flatten(), features_to_plot, colors):
    sns.histplot(df[feat], kde=True, ax=ax, color=color, bins=35)
    ax.set_title(f"Distribution of {feat}", fontweight='bold')
    ax.set_xlabel(feat)
plt.tight_layout()
plt.show()


In [ ]:
# Plot 2: Correlation Matrix between Audio Features
plt.figure(figsize=(10, 8))
corr = df[AUDIO_FEATURES].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.5)
plt.title("Correlation Matrix of Spotify Acoustic Audio Features", fontsize=14, fontweight='bold')
plt.show()


In [ ]:
# Plot 3: Acoustic Landscape (Danceability vs Energy colored by Valence)
plt.figure(figsize=(11, 7))
scatter = plt.scatter(
    df['Danceability'], df['Energy'],
    c=df['Valence'], cmap='viridis',
    alpha=0.6, edgecolors='none', s=25
)
cbar = plt.colorbar(scatter)
cbar.set_label('Valence (Musical Positivity / Happiness)', rotation=270, labelpad=20)
plt.title("Danceability vs. Energy Landscape (Colored by Valence)", fontsize=14, fontweight='bold')
plt.xlabel("Danceability")
plt.ylabel("Energy")
plt.show()


## 4. Sonic Vibe Clustering (K-Means)


In [ ]:
# Group tracks into 5 intuitive acoustic vibes using K-Means
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['vibe_cluster'] = kmeans.fit_predict(df[norm_cols])

VIBE_LABELS = {
    0: "⚡ High-Energy Dance & Workout",
    1: "🌿 Chill Acoustic & Coffeehouse",
    2: "🌙 Late Night Melancholic & Ambient",
    3: "🎉 Upbeat Pop & Anthems",
    4: "🧘 Focus Lo-Fi & Instrumental"
}
df['vibe_name'] = df['vibe_cluster'].map(VIBE_LABELS)

print("Vibe Cluster Distribution:")
print(df['vibe_name'].value_counts())


## 5. Content-Based Recommendation Engine (Acoustic Matching)


In [ ]:
# Fit Nearest Neighbors on the 9 normalized acoustic dimensions
feature_matrix = df[norm_cols].to_numpy(dtype=np.float32)

content_knn = NearestNeighbors(n_neighbors=50, metric='cosine', algorithm='brute')
content_knn.fit(feature_matrix)

def recommend_content_based(track_idx, n_recommendations=5, filter_same_artist=False, vibe_filter=None):
    query_vec = feature_matrix[track_idx].reshape(1, -1)
    distances, indices = content_knn.kneighbors(query_vec, n_neighbors=n_recommendations * 3 + 1)

    cand_indices = indices.flatten()[1:]
    cand_distances = distances.flatten()[1:]
    seed_artist = df.loc[track_idx, 'Artist']

    results = []
    for idx, dist in zip(cand_indices, cand_distances):
        row = df.loc[idx]
        if filter_same_artist and row['Artist'] == seed_artist:
            continue
        if vibe_filter and row['vibe_name'] != vibe_filter:
            continue
        sim_score = max(0.0, 1.0 - float(dist))
        results.append({
            'Artist': row['Artist'],
            'Track': row['Track'],
            'Vibe': row['vibe_name'],
            'Similarity': round(sim_score, 4),
            'Danceability': round(row['Danceability'], 3),
            'Energy': round(row['Energy'], 3),
            'Valence': round(row['Valence'], 3)
        })
        if len(results) >= n_recommendations:
            break
    return pd.DataFrame(results)

# Test Content-Based recommendation on Track 0
seed_idx = 0
print(f"Seed Track: '{df.loc[seed_idx, 'Track']}' by {df.loc[seed_idx, 'Artist']} [{df.loc[seed_idx, 'vibe_name']}]")
recommend_content_based(seed_idx, n_recommendations=5)


## 6. Collaborative Filtering via Latent Factor Modeling (TruncatedSVD)


In [ ]:
# Simulate realistic user interactions with taste profiles & power-law popularity
rng = np.random.default_rng(42)
n_users = 500
n_interactions = 15000

pop_scores = df['popularity_score'].values + 0.05
track_probs = pop_scores / pop_scores.sum()
vibe_clusters = df['vibe_cluster'].values

user_favs = {u: rng.choice(5, size=rng.integers(1, 3), replace=False) for u in range(1, n_users + 1)}
interactions = []
seen = set()

for u in range(1, n_users + 1):
    favs = user_favs[u]
    count = rng.integers(15, 45)
    for _ in range(count):
        if rng.random() < 0.7:
            cand = np.where(vibe_clusters == rng.choice(favs))[0]
            p = pop_scores[cand] / pop_scores[cand].sum()
            t_idx = rng.choice(cand, p=p)
        else:
            t_idx = rng.choice(len(df), p=track_probs)
        if (u, t_idx) not in seen:
            seen.add((u, t_idx))
            interactions.append({'user_id': u, 'track_index': t_idx, 'rating': rng.choice([3, 4, 5])})

interactions_df = pd.DataFrame(interactions)
print(f"Synthesized {len(interactions_df)} interactions across {interactions_df['user_id'].nunique()} users.")


In [ ]:
# Build User-Item Sparse Matrix
n_tracks = len(df)
user_map = {uid: i for i, uid in enumerate(sorted(interactions_df['user_id'].unique()))}
rows = [user_map[u] for u in interactions_df['user_id']]
cols = interactions_df['track_index'].values
data = interactions_df['rating'].values

sparse_user_item = csr_matrix((data, (rows, cols)), shape=(len(user_map), n_tracks), dtype=np.float32)

# Decompose via TruncatedSVD for latent factor representation
svd = TruncatedSVD(n_components=25, random_state=42)
user_factors = svd.fit_transform(sparse_user_item)
item_factors = svd.components_.T
print(f"Explained variance ratio sum: {svd.explained_variance_ratio_.sum():.4f}")


## 7. The Tri-Brid Recommendation Engine


In [ ]:
def hybrid_recommend(seed_track_idx, alpha=0.55, beta=0.30, gamma=0.15, n=5, vibe_filter=None):
    # 1. Acoustic similarity
    seed_vec = feature_matrix[seed_track_idx].reshape(1, -1)
    acoustic_sims = cosine_similarity(seed_vec, feature_matrix).flatten()

    # 2. Collaborative item-item latent similarity
    item_vec = item_factors[seed_track_idx].reshape(1, -1)
    collab_sims = (cosine_similarity(item_vec, item_factors).flatten() + 1.0) / 2.0

    # 3. Popularity
    pop_scores = df['popularity_score'].to_numpy()

    # Weighted composite score
    total_w = alpha + beta + gamma
    hybrid_score = (alpha * acoustic_sims + beta * collab_sims + gamma * pop_scores) / total_w
    hybrid_score[seed_track_idx] = -1.0

    ranked = np.argsort(hybrid_score)[::-1]
    results = []
    seed_artist = df.loc[seed_track_idx, 'Artist']

    for idx in ranked:
        row = df.loc[idx]
        if vibe_filter and row['vibe_name'] != vibe_filter:
            continue
        results.append({
            'Artist': row['Artist'],
            'Track': row['Track'],
            'Vibe': row['vibe_name'],
            'Hybrid Score': round(float(hybrid_score[idx]), 4),
            'Acoustic Match': round(float(acoustic_sims[idx]), 4),
            'Collab Match': round(float(collab_sims[idx]), 4),
            'Popularity': round(float(pop_scores[idx]), 4)
        })
        if len(results) >= n:
            break
    return pd.DataFrame(results)

# Test Hybrid Recommender
print(f"Seed Track: '{df.loc[seed_idx, 'Track']}' by {df.loc[seed_idx, 'Artist']}")
hybrid_recommend(seed_idx, alpha=0.55, beta=0.30, gamma=0.15, n=5)


## 8. Quantitative Evaluation & Benchmark Suite
We evaluate models using:
* **Precision@K & Recall@K**: Relevant items captured in top-K recommendations.
* **MAP@K (Mean Average Precision)**: Position-weighted hit ranking.
* **NDCG@K**: Discounted cumulative gain reflecting user satisfaction.
* **Intra-List Diversity (ILD)**: Pairwise acoustic distance across recommendations.
* **Catalog Novelty**: Exploration of the catalog beyond head/blockbuster hits.


In [ ]:
from src.evaluation import run_benchmark

print("Running benchmark across Content-Based, Collaborative, and Hybrid engines...")
benchmark_results = run_benchmark(df, interactions_df, k=10)
benchmark_results


In [ ]:
# Benchmark Visualization
fig, ax = plt.subplots(figsize=(11, 6))
benchmark_plot_df = benchmark_results.set_index('Model')[['Precision@10', 'MAP@10', 'NDCG@10', 'Intra-List Diversity']]
benchmark_plot_df.plot(kind='bar', ax=ax, colormap='viridis', edgecolor='black')
plt.title("Recommender Systems Benchmark Comparison (K=10)", fontsize=14, fontweight='bold')
plt.ylabel("Score")
plt.xticks(rotation=15, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


## 9. Conclusion & Key Takeaways
1. **Hybrid Outperforms Single Models:** By combining acoustic cosine similarity with SVD latent interaction factors and popularity regularization, the Hybrid model delivers superior **MAP@10** and **NDCG@10** while preserving high **Intra-List Diversity**.
2. **Cold-Start Resilience:** New songs with 0 interactions still receive high-quality recommendations through the Content-Based audio branch.
3. **Interactive Deployment:** An interactive web application is available via `streamlit run app.py`, allowing live search, radar charts, and real-time vibe filtering across all 20,000+ tracks.
